In [2]:
!pip install loompy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 27.4 kB/s eta 0:00:00 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 60.4 kB/s eta 0:00:00 0:00:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 56.7 kB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for loompy: filename=loompy-3.0.8-py3-none-any.whl size=54013 sha256=f37c9a05122ad702ed2774896b4694f0ddb5d920278113add926f14c712ea118
  Stored in directory: /root/.cache/pip/wheels/7a/b0/51/0054e104762c74124646fb54c3748d0fddd8efb7d5d5514464
  Created wheel for numpy-groupies: filename=numpy_groupies-0.9.22-py3-none-any.whl size=25846 sha256=6156323471e158f21f044c76d96cc8d27c2cec2c6ec9bb5b28d5b42e2b8b3546
  Stored in directory: /root/.cache/pip/wheels/2e/b9/5a/225e71b783e29f2098ba2dc8a5266f02b2d0d08f1890a28548
Successfully built loompy numpy-groupies


In [ ]:
#input: Allen institute h5ad and metadata 
#output: subset of allen institute data

In [1]:
import pandas as pd
from samalg import SAM
import scanpy as sc
import statistics
import matplotlib.pyplot as plt
import seaborn as sns
import random
import pandas as pd
import matplotlib.colors
import scipy
import numpy as np
import sklearn.metrics as metrics
from scipy import sparse
import time
import loompy
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import pickle

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#a dictionary which contains the heirarchical relationships between subclasses in the allen institute data
with open('parent_dict.pkl', 'rb') as f:
    parent_dict = pickle.load(f)

In [5]:
#raw allen institute data and subclass labels
dat = sc.read_h5ad('AIT17.0.rawcount.h5ad', backed = 'r')
metadata = pd.read_csv('Allen_institute_labels_v2_classsubclasssupertype.csv')
print('loaded data')

ncl = []
for item in dat.obs_names:
    split = item.split('-')
    ncl.append(split[0] + split[1])
dat.obs_names = ncl
print('renaming obs')

nlc_meta = []
for item in metadata.index:
    nlc_meta.append(metadata.loc[item, 'cell_barcode'] + metadata.loc[item, 'library_label'])
metadata.index = nlc_meta

loaded data
renaming obs


In [ ]:
for item in metadata['supertype_id_label'].unique():
    if item not in parent_dict:
        parent_dict[item] = 'not hypo'
        print(item)

In [6]:
subdat_dict = {}
level = 'subclass_id_label'
ct = metadata[level].unique()
for item in ct:
    subdat_dict[item] = list(metadata[metadata.loc[:,level] == item].index)

In [7]:
for item in subdat_dict.keys():
    if item not in parent_dict:
        parent_dict[item] = 'not hypo'

In [8]:
fin_obs = set(dat.obs_names) & set(metadata.index)
metadata = metadata.loc[fin_obs, :]

In [6]:
level = 'subclass_id_label'
HYPO = [i for i in metadata[level].unique() if parent_dict[parent_dict[i]] == 'hypo']

In [20]:
names = []
num_cells = 0
subset = False
for item in subdat_dict.keys():
    if item in HYPO:
        if len(subdat_dict[item]) > num_cells and subset:
            rsd = list(random.sample(subdat_dict[item], num_cells))
        else:
            rsd = subdat_dict[item]
        names = names + list(set(rsd))
print(len(names))
findat = dat[dat.obs_names.isin(names)]
findat.write_loom('')

dat_two = sc.read_loom('')
dat_two.obs_names = dat_two.obs['obs_names']
print(dat_two.obs_names)
dat_two.obs = metadata.loc[dat_two.obs_names,:]
print(dat_two.obs_names)
dat_two.write_loom('')

86232
